Recommended to use virtual env:

1. python -m venv .venv
2. .venv\Scripts\activate
3. pip install -r requirements.txt

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as pyplot

df = pd.read_csv('Fifa_world_cup_matches.csv', encoding='latin-1')

In [ ]:
# Adds new 'result' column. 1 = Win, 0 = Loss
# Tracks whether Team 1 won or not.
goals_team_1 = pd.to_numeric(df['number of goals team1'])
goals_team_2 = pd.to_numeric(df['number of goals team2'])
df['result'] = (goals_team_1 > goals_team_2).astype(int)

# Stage Feature, Organized Group Stage and Knockouts (incl. quarters, semis, finals etc..)
df['stage_type'] = np.where(df['category'].str.startswith('Group'), 'Group', 'Knockout')

# Unneccesary Columns
df = df.drop(columns = ['team1', 'team2', 'date', 'hour']) 

# Columns that give away the result of the match
df = df.drop(columns = [
    'number of goals team1','number of goals team2',
    'conceded team1','conceded team2',
    'penalties scored team1','penalties scored team2',
    'goal inside the penalty area team1','goal inside the penalty area team2',
    'goal outside the penalty area team1','goal outside the penalty area team2',
    'own goals team1','own goals team2',
    'assists team1','assists team2'
])


In [ ]:
# Looks for cells with % in them and converts them into decimal (eg 56% -> 0.56)
percent_cols = [c for c in df.columns if df[c].astype(str).str.contains('%', na=False).any()]
if percent_cols:
    df[percent_cols] = df[percent_cols].apply(
        lambda col: col.astype(str).str.replace(r'[%\s]', '', regex=True).replace('', pd.NA).astype(float) / 100
    )

In [ ]:
# Calculate differences
import re

x_temp = df.iloc[:, :-1] # Selects all but the result column

team1_cols = x_temp.filter(like="team1").columns
team2_cols = x_temp.filter(like="team2").columns

def base_name(col):
    return re.sub(r'(?i)\s*team1\s*$', '', col).strip()

diff_features = pd.DataFrame(index=x_temp.index)
diff_col_names = [f"{base_name(c)} diff" for c in team1_cols]

for i in range(len(team1_cols)):
    col1 = team1_cols[i]
    col2 = team2_cols[i]
    diff_col = diff_col_names[i]

    diff_features[diff_col] = x_temp[col1] - x_temp[col2]

In [ ]:
# FINAL DATAFRAME
final_df = diff_features.copy()
final_df['result'] = df.loc[diff_features.index, 'result']
final_df['is_knockout'] = (df.loc[diff_features.index, 'stage_type'] == 'Knockout').astype(int)

X = final_df.drop(columns=['result'])
y = final_df['result']

In [ ]:
from sklearn.model_selection import train_test_split
# 20% Test / 80% Train
X_train, X_test, y_train, y_test = train_test_split(X,y, random_state = 17, test_size = 0.2)

## Logistic Regression

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix
from sklearn.model_selection import StratifiedKFold, cross_val_score

print("Class distribution (train):", y_train.value_counts(normalize=True).to_dict())

log_reg_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(
        max_iter=500,
        class_weight="balanced",
        C=1.0,
        solver="lbfgs",
        random_state=42
    ))
])

log_reg_pipe.fit(X_train, y_train)
y_pred_lr = log_reg_pipe.predict(X_test)
y_prob_lr = log_reg_pipe.predict_proba(X_test)[:, 1]

print("LogReg accuracy:", accuracy_score(y_test, y_pred_lr))
print("LogReg ROC AUC:", roc_auc_score(y_test, y_prob_lr))
print(classification_report(y_test, y_pred_lr))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred_lr))

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_auc = cross_val_score(log_reg_pipe, X, y, cv=cv, scoring="roc_auc")
print("CV ROC AUC mean:", cv_auc.mean(), "std:", cv_auc.std())

### Extract Coefficients

In [ ]:
log_reg = log_reg_pipe.named_steps['clf']
coef = log_reg.coef_[0]

features = pd.DataFrame({
    'feature': X_train.columns,
    'coefficient': coef,
    'abs_value': np.abs(coef)
}).sort_values(by='abs_value', ascending=False)

print(features.head(15))

### Visualize the findings

In [ ]:
top_features = features.head(10)

pyplot.figure()
pyplot.barh(top_features["feature"], top_features["abs_value"], color="skyblue")
pyplot.xlabel("Coefficient Value")
pyplot.title("Top 10 Contributing Features to Match Outcome Prediction using LogReg")
pyplot.gca().invert_yaxis()
pyplot.tight_layout()
pyplot.show()

### LogReg Interpretation:

**Class Distribution on training data** <br>
Accuracy if you always assumed lose/win for a given matchup, naively: <br>
Class 0 (Loss): ~58% <br>
Class 1 (Win): ~41% <br>

**ROC AUC (Receiver Operating Characteristics Area Under the Curve):** <br>
Probability the model ranks a random given Win, higher than a random given loss. Basically a metric to show how well it performed at predicting the result. 1 is perfect, 0.5 is like guessing randomly: <br>
<br>
ROC AUC: 0.725 (Can be determined as useful)<br>

**Cross Validation (CV) AUC Mean/Std** <br>
ROC AUC Score on unseen data: <br> <br>
Mean: 0.71 <br>
Standard Deviation: 0.05 <br>
(Very similar to the testing AUC ROC score. Low std shows score is stable across folds)

**LogReg Accuracy:** <br>
Predicts ~69% of match outcomes. <br>
| Class    | Precision | Recall | F1   |
|----------|-----------|--------|------|
| 0 (Lose) | 0.60      | 0.60   | 0.60 |
| 1 (Win)  | 0.75      | 0.75   | 0.75 |


... And is better at predicting a win rather than a loss.

**Confusion Matrix**
To support the LogReg accuracy:

| Actual   | Predicted 0 (Loss) | Predicted 1 (Win) |
|----------|--------------------|-------------------|
| 0 (Lose) | 3                  | 2                 |
| 1 (Win)  | 2                  | 6                 |



## XGBoost

In [ ]:
import xgboost as xgb
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer

print("Class distribution (train):", y_train.value_counts(normalize=True).to_dict())

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"scale_pos_weight (neg/pos): {scale_pos_weight:.2f}")

xgb_model = xgb.XGBClassifier(
    n_estimators=50,
    max_depth=2,
    learning_rate=0.1,
    subsample=0.6,
    colsample_bytree=0.6,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    eval_metric='auc',
    early_stopping_rounds=10
)

xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False
)

y_pred_xgb = xgb_model.predict(X_test)
y_prob_xgb = xgb_model.predict_proba(X_test)[:, 1]

print("\nXGBoost accuracy:", accuracy_score(y_test, y_pred_xgb))
print("XGBoost ROC AUC:", roc_auc_score(y_test, y_prob_xgb))
print(classification_report(y_test, y_pred_xgb))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred_xgb))

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
xgb_cv = xgb.XGBClassifier(
    n_estimators=50,
    max_depth=2,
    learning_rate=0.1,
    subsample=0.6,
    colsample_bytree=0.6,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    eval_metric='auc'
)
cv_auc = cross_val_score(xgb_cv, X, y, cv=cv, scoring="roc_auc")
print("\nCV ROC AUC mean:", cv_auc.mean(), "std:", cv_auc.std())

### Extract Importance Values

In [ ]:
xgb_importances = xgb_model.feature_importances_

xgb_features = pd.DataFrame({
    "feature": X_train.columns,
    "importance": xgb_importances,
    "abs_importance": np.abs(xgb_importances)
}).sort_values(by='abs_importance', ascending=False)

print(xgb_features.head(15))

### Visualize Findings

In [ ]:
top_features = xgb_features.head(10)

pyplot.figure()
pyplot.barh(top_features["feature"], top_features["abs_importance"], color="skyblue")
pyplot.xlabel("Importance Value")
pyplot.title("Top 10 Contributing Features to Match Outcome Prediction using XGBoost")
pyplot.gca().invert_yaxis()
pyplot.tight_layout()
pyplot.show()

## Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

print("Class distribution (train):", y_train.value_counts(normalize=True).to_dict())

rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=2, 
    min_samples_split=5,
    min_samples_leaf=2,  
    max_features='sqrt', 
    class_weight='balanced', 
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)
y_prob_rf = rf_model.predict_proba(X_test)[:, 1]

print("\nRandom Forest accuracy:", accuracy_score(y_test, y_pred_rf))
print("Random Forest ROC AUC:", roc_auc_score(y_test, y_prob_rf))
print(classification_report(y_test, y_pred_rf))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred_rf))

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_auc_rf = cross_val_score(rf_model, X, y, cv=cv, scoring="roc_auc")
print("\nCV ROC AUC mean:", cv_auc_rf.mean(), "std:", cv_auc_rf.std())




### Extract Important Features

In [ ]:
rf_importances = rf_model.feature_importances_

rf_features = pd.DataFrame({
    "feature": X_train.columns,
    "importance": rf_importances,
    "abs_importance": np.abs(rf_importances)
}).sort_values(by="abs_importance", ascending=False)

print(rf_features.head(15))

### Visualize Importance Ratings

Reminder: RF rates importance ratings as a percentage

In [ ]:
top_features = rf_features.head(10)

pyplot.figure()
pyplot.barh(top_features["feature"], top_features["abs_importance"], color="skyblue")
pyplot.xlabel("Importance Rating percentage")
pyplot.title("Top 10 Contributing Features to Match Outcome Prediction using Random Forest")
pyplot.gca().invert_yaxis()
pyplot.tight_layout()
pyplot.show()

# CatBoost

In [ ]:
from catboost import CatBoostClassifier

# Startup CatBoost
cat_model = CatBoostClassifier(
    iterations=500, learning_rate = 0.1, depth=6, eval_metric="Logloss",
    random_state=42, verbose=False,
    class_weights=[1, (y_train == 0).sum() / (y_train == 1).sum()]
)

# Fit
cat_model.fit(X_train, y_train, eval_set=(X_test, y_test), use_best_model=True)

# Predict
y_pred_cat = cat_model.predict(X_test)
y_prob_cat = cat_model.predict_proba(X_test)[:, 1]

# Evaluate
print("\nCatBoost accuracy:", accuracy_score(y_test, y_pred_cat))
print("CatBoost ROC AUC:", roc_auc_score(y_test, y_prob_cat))
print(classification_report(y_test, y_pred_cat))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred_cat))

### Extract Features

In [ ]:
cat_importances = cat_model.get_feature_importance()

cat_features = pd.DataFrame({
    "feature": X_train.columns,
    "importance": cat_importances,
    "abs_importance": np.abs(cat_importances)
}).sort_values(by='abs_importance', ascending=False)

print(cat_features.head(15))

### Visualize Findings

In [ ]:
top_features = cat_features.head(10)

pyplot.figure()
pyplot.barh(top_features["feature"], top_features["abs_importance"], color="skyblue")
pyplot.xlabel("Importance Rating")
pyplot.title("Top 10 Contributing Features to Match Outcome Prediction using CatBoost")
pyplot.gca().invert_yaxis()
pyplot.tight_layout()
pyplot.show()